
###### 14_agentic_rag

###### Purpose

The purpose of this notebook is to replace deterministic rule-based routing with LLM-driven tool selection to demonstrate Agentic Retrieval-Augmented Generation (Agentic RAG).


###### Technologies Used

- Databricks

- Delta Lake

- Unity Catalog

- Databricks Vector Search

- Databricks Embedding Foundation Model (databricks-gte-large-en)

- LLM Model (databricks-meta-llama-3-1-8b-instruct)

- Python

- PySpark

- Databricks SDK

- Rule-Based Routing


###### Input

-  User question

-  Existing Vector Search endpoint and index

-  Customer-notes Delta table

-  Customer-note embeddings

-  Embedding and LLM endpoint configuration


######  Output

- LLM generated grounded response


######  Architecture

```text

Question
   ↓
Agent (Decision)
   ↓
Tool
   ↓
Large Language Model (LLM)
   ↓
Grounded Response

```


Differece between fixed RAG and Agent selected tools:


Fixed RAG:

``` text

Question
    ↓
Not a count question
    ↓
Vector Search Tool
    ↓
Similarity Search
    ↓
Top 3 Notes

```

Agent selected tools:


``` text

Question
      ↓
LLM Agent
      ↓
Tool Selection
      ↓
Execute Tool
      ↓
Tool Result
      ↓
LLM
      ↓
Grounded Answer

```


###### Section 0 : Install Vector Search client

In [0]:
%pip install databricks-vectorsearch
dbutils.library.restartPython()

###### Section 1 :  Load Project Configuration

In [0]:
%run ./00_project_config


###### Section 2 : Import Libraries and Initialize Clients

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.vector_search.client import VectorSearchClient
from databricks.sdk.service.serving import ChatMessage, ChatMessageRole

w = WorkspaceClient()
vsc = VectorSearchClient(disable_notice=True)
vsc.list_endpoints()


###### Section 3 : Connect to Existing Vector Search Index

In [0]:
index = vsc.get_index(
    endpoint_name = VECTOR_SEARCH_ENDPOINT_NAME,
    index_name= VECTOR_INDEX_NAME
)

index_description = index.describe()

print("Connected to the existing Vector Search index.")
print(
    "Index state:",
    index_description
    .get("status", {})
    .get("detailed_state", "UNKNOWN")
)

###### Section 4 : Define LLM Response Helper

In [0]:
def generate_answer(prompt: str):
    response = w.serving_endpoints.query(
        name=LLM_MODEL,
        messages=[
            ChatMessage(
                role=ChatMessageRole.USER,
                content=prompt
            )
        ],
        max_tokens=300,
        temperature=0.0
    )

    if (
        not response.choices
        or response.choices[0].message is None
    ):
        raise ValueError(
            "The LLM returned no response."
        )

    return response.choices[0].message.content

###### Section 5 : Define Vector Search Tool

In [0]:
def search_customer_notes(
    question: str,
    num_results: int = 3
):
    if not question or not question.strip():
        raise ValueError("Question cannot be empty.")

    if num_results <= 0:
        raise ValueError(
            "num_results must be greater than zero."
        )

    response = w.serving_endpoints.query(
        name=EMBEDDING_MODEL,
        input=[question]
    )

    if (
        not response.data
        or response.data[0].embedding is None
    ):
        raise ValueError(
            "The embedding model returned no embedding."
        )

    question_embedding = [
        float(value)
        for value in response.data[0].embedding
    ]

    results = index.similarity_search(
        query_vector=question_embedding,
        columns=["customer_id", "note"],
        num_results=num_results
    )

    rows = (
        results
        .get("result", {})
        .get("data_array", [])
    )

    if not rows:
        return {
            "tool": "search_customer_notes",
            "status": "no_results",
            "context": "",
            "rows": [],
            "result_count": 0
        }

    context_lines = [
        f"Customer {int(customer_id)}: {note}"
        for customer_id, note, score in rows
    ]

    return {
        "tool": "search_customer_notes",
        "status": "success",
        "context": "\n".join(context_lines),
        "rows": rows,
        "result_count": len(rows)
    }

###### Section 6 : Define SQL Analytics Tool

In [0]:
def count_customer_notes():
    count = spark.table(NOTES_TABLE).count()

    return {
        "tool": "count_customer_notes",
        "status": "success",
        "count": count
    }

###### Section 7 : Tool execution

In [0]:
VALID_TOOLS = {
    "count_customer_notes",
    "search_customer_notes"
}


def choose_tool(question: str) -> str:
    if not question or not question.strip():
        raise ValueError("Question cannot be empty.")

    tool_prompt = f"""
You are a tool-routing agent.

Choose the best tool for the user's question.

Available tools:

1. count_customer_notes
Use for questions about counts, totals, how many, or the number
of customer notes.

2. search_customer_notes
Use for questions about reasons, complaints, dissatisfaction,
cancellation, customer sentiment, or other semantic information
contained in customer notes.

Question:
{question}

Return exactly one tool name and nothing else:

count_customer_notes
or
search_customer_notes
"""

    tool_name = (
        generate_answer(tool_prompt)
        .strip()
        .lower()
        .replace("`", "")
        .replace('"', "")
        .replace("'", "")
        .replace(".", "")
        .strip()
    )

    if tool_name not in VALID_TOOLS:
        raise ValueError(
            f"LLM returned an unsupported tool: {tool_name}"
        )

    return tool_name

###### Section 8 : RAG_Agent

In [0]:
def agentic_rag_agent(question: str):
    if not question or not question.strip():
        raise ValueError("Question cannot be empty.")

    # Step 1: Let the LLM select the appropriate tool
    selected_tool = choose_tool(question)

    print(f"Tool selected by LLM: {selected_tool}")

    # Step 2: Execute the selected tool
    if selected_tool == "count_customer_notes":
        tool_result = count_customer_notes()

        if tool_result["status"] != "success":
            raise RuntimeError(
                f"Count tool failed: {tool_result}"
            )

        final_prompt = f"""
You are a telecom customer-support assistant.

Use only the tool result below to answer the user's question.
Do not add information that is not present in the tool result.

Question:
{question}

Tool used:
count_customer_notes

Tool result:
Customer-note count: {tool_result["count"]}

Provide a concise and clear final answer.
"""

    elif selected_tool == "search_customer_notes":
        tool_result = search_customer_notes(question)

        if tool_result["status"] == "no_results":
            return {
                "selected_tool": selected_tool,
                "tool_result": tool_result,
                "answer": (
                    "I don't have enough information from "
                    "the retrieved customer notes."
                )
            }

        if tool_result["status"] != "success":
            raise RuntimeError(
                f"Vector Search tool failed: {tool_result}"
            )

        context = tool_result["context"]

        final_prompt = f"""
You are a telecom customer-support assistant.

Answer the user's question using ONLY the retrieved customer notes.
Do not use outside knowledge or make assumptions.

If the retrieved notes do not contain enough information,
respond exactly:

"I don't have enough information from the retrieved customer notes."

Retrieved Customer Notes:
{context}

Question:
{question}

Provide a concise and factual answer.
"""

    else:
        raise ValueError(
            f"Unsupported selected tool: {selected_tool}"
        )

    # Step 3: Generate the final user-facing answer
    final_answer = generate_answer(final_prompt)

    # Step 4: Return a structured trace
    return {
        "selected_tool": selected_tool,
        "tool_result": tool_result,
        "answer": final_answer
    }

###### Section 8 :  End-to-End Tests

In [0]:
test_questions = [
    "How many customer notes are there?",
    "Why are customers cancelling service?",
    "Are there any billing complaints?",
    "What upgrades are customers requesting?"
]

for question in test_questions:
    result = agentic_rag_agent(question)

    print("=" * 80)

    print("QUESTION:")
    print(question)

    print("\nSELECTED TOOL:")
    print(result["selected_tool"])

    print("\nTOOL STATUS:")
    print(result["tool_result"]["status"])

    print("\nFINAL ANSWER:")
    print(result["answer"])

    print()

###### Notebook Summary

- Get Configurations for Vector Search endpoint name,  Vector index name,  Embedding model name and LLM endpoint name.

- Connect to Vector Search Index.

- Created reusable functions:

    -  Vector Search Tool

    -  SQL Analytics Tool

    -  LLM Response Helper

    -  Tool Selection Function

    -  Agentic RAG Workflow

- Test Agent 

###### Key Learnings

- Used Databricks Vector Search to retrieve semantically similar historical customer notes info.

- Developed Agent router to  dynamically select the appropriate tool based on the user's question.

- LLMs can dynamically choose tools instead of relying on hard-coded routing rules.

###### Notebook Conclusion

- This notebook demonstrated how an LLM can act as an agent that selects the most appropriate tool for each question. Compared with rule-based routing, the LLM provides more flexible tool selection while still generating grounded answers using tool outputs.

###### Next Notebook

15_evaluation

The purpose of this notebook is to evaluate - retrieval qualit,  answer relevance,  groundedness and hallucination risk for Agentic RAG response.